# LAB 4 — DeepEval: Testes Unitários para o Pipeline RAG Jurídico
## Aula 5 — Avaliação de Pipelines RAG

Escreve **5 testes unitários** com **DeepEval** validando o pipeline RAG da Aula 5: Faithfulness, AnswerRelevancy, Hallucination, Toxicity e Bias.

**Stack:**
- **LLM judge DeepEval**: Groq `llama-3.1-8b-instant` (fallback Ollama `llama3.2:3b`) via wrapper `DeepEvalBaseLLM`
- **Embeddings**: Ollama `bge-m3` (1024d)
- **Vector store**: OpenSearch 3.x (mesmo índice TCU 2026)

**Pré-requisito:** LAB 1 e LAB 2 executados.


## 1. Instalação


In [ ]:
import subprocess, sys
for pkg in ['deepeval>=0.21', 'pandas', 'python-dotenv>=1.0',
            'langchain>=0.2', 'langchain-core>=0.2',
            'langchain-groq>=0.1', 'langchain-ollama>=0.1',
            'opensearch-py>=2.7']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('dependencias instaladas')


## 2. Stack (Groq + Ollama + OpenSearch)


In [ ]:
import os, json, warnings
from pathlib import Path
from dotenv import load_dotenv
warnings.filterwarnings('ignore')

for env_path in [Path.cwd()/'.env', Path.home()/'mba-rag'/'.env',
                  Path.cwd().parent.parent/'.env']:
    if env_path.exists():
        load_dotenv(env_path, override=True); break

GROQ_API_KEY       = os.getenv('GROQ_API_KEY', '')
GROQ_LLM_MODEL     = os.getenv('GROQ_LLM_MODEL',   'llama-3.1-8b-instant')
OLLAMA_BASE_URL    = os.getenv('OLLAMA_BASE_URL',    'http://localhost:11434')
OLLAMA_LLM_MODEL   = os.getenv('OLLAMA_LLM_MODEL',   'llama3.2:3b')
OLLAMA_EMBED_MODEL = os.getenv('OLLAMA_EMBED_MODEL', 'bge-m3')
OPENSEARCH_HOST    = os.getenv('OPENSEARCH_HOST',     'localhost')
OPENSEARCH_PORT    = int(os.getenv('OPENSEARCH_PORT', 9200))
OPENSEARCH_USER    = os.getenv('OPENSEARCH_USER',     'admin')
OPENSEARCH_PASS    = os.getenv('OPENSEARCH_PASSWORD', 'admin')
INDEX_NAME         = os.getenv('INDEX_NAME', 'corpus_juridico_aula4')

from langchain_groq import ChatGroq
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.messages import HumanMessage
from opensearchpy import OpenSearch

llm = None; LLM_PROVIDER = None; LLM_MODEL = None
if GROQ_API_KEY:
    try:
        c = ChatGroq(model=GROQ_LLM_MODEL, api_key=GROQ_API_KEY,
                     temperature=0.0, max_tokens=1024, timeout=30)
        c.invoke([HumanMessage(content='ok')])
        llm, LLM_PROVIDER, LLM_MODEL = c, 'groq', GROQ_LLM_MODEL
    except Exception:
        pass
if llm is None:
    llm = ChatOllama(model=OLLAMA_LLM_MODEL, base_url=OLLAMA_BASE_URL,
                     temperature=0.0, num_predict=1024)
    LLM_PROVIDER, LLM_MODEL = 'ollama', OLLAMA_LLM_MODEL

embeddings = OllamaEmbeddings(model=OLLAMA_EMBED_MODEL, base_url=OLLAMA_BASE_URL)
_ = embeddings.embed_query('teste'); assert len(_) == 1024

os_client = OpenSearch(
    hosts=[{'host': OPENSEARCH_HOST, 'port': OPENSEARCH_PORT}],
    http_auth=(OPENSEARCH_USER, OPENSEARCH_PASS),
    use_ssl=False, verify_certs=False, timeout=60,
)
print(f'Stack OK | LLM={LLM_PROVIDER}/{LLM_MODEL} | embed={OLLAMA_EMBED_MODEL} | OS={os_client.info()["version"]["number"]}')


## 3. Wrapper `DeepEvalBaseLLM` para Groq/Ollama

DeepEval exige um modelo que herde de `DeepEvalBaseLLM`. Criamos um wrapper sobre a LLM ativa (Groq ou Ollama).


In [ ]:
from deepeval.models import DeepEvalBaseLLM
from langchain_core.messages import HumanMessage

class LangchainJudge(DeepEvalBaseLLM):
    """DeepEval LLM judge que delega para qualquer LangChain ChatModel (Groq/Ollama)."""
    def __init__(self, lc_model, model_name: str):
        self.lc_model = lc_model
        self._name    = model_name
    def load_model(self):
        return self.lc_model
    def generate(self, prompt: str) -> str:
        return self.lc_model.invoke([HumanMessage(content=prompt)]).content
    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)
    def get_model_name(self) -> str:
        return self._name

judge = LangchainJudge(llm, f'{LLM_PROVIDER}/{LLM_MODEL}')
print(f'DeepEval LLM judge: {judge.get_model_name()}')


## 4. Pipeline RAG (carregar dataset + função de execução)


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

DATASET_LAB1 = 'dataset_avaliacao_completo.json'
if not os.path.exists(DATASET_LAB1):
    raise FileNotFoundError(f'{DATASET_LAB1} nao encontrado. Execute o LAB 1 primeiro.')
with open(DATASET_LAB1, encoding='utf-8') as f:
    dataset = json.load(f)
print(f'{len(dataset)} pares carregados')

PROMPT = ChatPromptTemplate.from_messages([
    ('system', 'Voce e um assistente juridico brasileiro. Responda APENAS com base nos trechos.'),
    ('human', 'TRECHOS:\n{contextos}\n\nPERGUNTA: {pergunta}\n\nRESPOSTA:')
])
CHAIN = PROMPT | llm | StrOutputParser()

def executar_pipeline(question: str, k: int = 5):
    v = embeddings.embed_query(question)
    r = os_client.search(index=INDEX_NAME,
                          body={'size': k,
                                'query': {'knn': {'embedding': {'vector': v, 'k': k}}},
                                '_source': ['titulo','conteudo']})
    ctxs = [(h['_source'].get('titulo','') + '\n' + h['_source'].get('conteudo','')).strip()
             for h in r['hits']['hits']]
    bloco = '\n\n'.join(f'[Doc {i+1}]\n{c[:600]}' for i, c in enumerate(ctxs))
    ans = CHAIN.invoke({'contextos': bloco, 'pergunta': question}).strip()
    return ans, ctxs

p0 = dataset[0]
_a, _c = executar_pipeline(p0['question'])
print(f'smoke OK | {len(_c)} ctxs | answer: {_a[:120]}...')


## 5. Teste 1 — FaithfulnessMetric (>= 0.80)


In [ ]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric

p = dataset[0]
ans, ctxs = executar_pipeline(p['question'])
case = LLMTestCase(input=p['question'], actual_output=ans, retrieval_context=ctxs,
                    expected_output=p['ground_truth'])
metric = FaithfulnessMetric(threshold=0.80, model=judge, include_reason=True)
metric.measure(case)
print(f'Faithfulness: {metric.score:.4f} (threshold 0.80) - {"PASS" if metric.is_successful() else "FAIL"}')
print(f'Razao: {metric.reason[:240]}...')


## 6. Teste 2 — AnswerRelevancyMetric (>= 0.75)


In [ ]:
from deepeval.metrics import AnswerRelevancyMetric
metric_rel = AnswerRelevancyMetric(threshold=0.75, model=judge, include_reason=True)
metric_rel.measure(case)
print(f'AnswerRelevancy: {metric_rel.score:.4f} - {"PASS" if metric_rel.is_successful() else "FAIL"}')
print(f'Razao: {metric_rel.reason[:240]}...')


## 7. Teste 3 — HallucinationMetric (<= 0.20)


In [ ]:
from deepeval.metrics import HallucinationMetric
case_hall = LLMTestCase(input=p['question'], actual_output=ans, context=ctxs)
metric_hall = HallucinationMetric(threshold=0.20, model=judge, include_reason=True)
metric_hall.measure(case_hall)
print(f'Hallucination: {metric_hall.score:.4f} (max 0.20) - {"PASS" if metric_hall.is_successful() else "FAIL"}')
print(f'Razao: {metric_hall.reason[:240]}...')


## 8. Teste 4 — ToxicityMetric (<= 0.10)


In [ ]:
from deepeval.metrics import ToxicityMetric
metric_tox = ToxicityMetric(threshold=0.10, model=judge, include_reason=True)
metric_tox.measure(case)
print(f'Toxicity: {metric_tox.score:.4f} (max 0.10) - {"PASS" if metric_tox.is_successful() else "FAIL"}')
print(f'Razao: {metric_tox.reason[:240]}...')


## 9. Teste 5 — BiasMetric (<= 0.20)


In [ ]:
from deepeval.metrics import BiasMetric
metric_bias = BiasMetric(threshold=0.20, model=judge, include_reason=True)
metric_bias.measure(case)
print(f'Bias: {metric_bias.score:.4f} (max 0.20) - {"PASS" if metric_bias.is_successful() else "FAIL"}')
print(f'Razao: {metric_bias.reason[:240]}...')


## 10. Bateria Completa em 5 Pares — Tabela Comparativa


In [ ]:
import pandas as pd
rows = []
N = 5
for i, par in enumerate(dataset[:N], 1):
    ans, ctxs = executar_pipeline(par['question'])
    c1 = LLMTestCase(input=par['question'], actual_output=ans, retrieval_context=ctxs,
                      expected_output=par['ground_truth'])
    c2 = LLMTestCase(input=par['question'], actual_output=ans, context=ctxs)
    ms = {
        'faithfulness':     FaithfulnessMetric(threshold=0.80, model=judge),
        'answer_relevancy': AnswerRelevancyMetric(threshold=0.75, model=judge),
        'hallucination':    HallucinationMetric(threshold=0.20, model=judge),
        'toxicity':         ToxicityMetric(threshold=0.10, model=judge),
        'bias':             BiasMetric(threshold=0.20, model=judge),
    }
    row = {'par': i, 'tipo': par.get('tipo','')}
    for nm, mt in ms.items():
        try:
            mt.measure(c2 if nm == 'hallucination' else c1)
            row[nm] = round(mt.score, 4)
            row[f'{nm}_ok'] = mt.is_successful()
        except Exception as e:
            row[nm] = 0.0; row[f'{nm}_ok'] = False
            print(f'  par {i}/{nm}: {e}')
    rows.append(row)
    print(f'  par {i} avaliado')

df = pd.DataFrame(rows)
print('\n=== Resultados ===')
print(df.to_string(index=False))

OUT = 'lab4_deepeval_resultados.csv'
df.to_csv(OUT, index=False, encoding='utf-8')
print(f'\n{OUT} salvo')

print('\n=== Resumo (pares aprovados por metrica) ===')
for m in ['faithfulness','answer_relevancy','hallucination','toxicity','bias']:
    ok = df[f'{m}_ok'].sum()
    print(f'  {m:<22} {ok}/{N}')


## 11. Padrão `pytest` para CI/CD

Em produção, transforme as métricas em **testes pytest** que rodam no pipeline de CI antes de cada deploy. Use `assert_test()` do DeepEval.


In [ ]:
from textwrap import dedent
_arquivo = 'test_pipeline_juridico.py'
_codigo = dedent('''
import pytest
from deepeval import assert_test
from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric

# importe seu LangchainJudge (Groq/Ollama) e executar_pipeline aqui:
# from meu_app import judge, executar_pipeline

@pytest.mark.parametrize("question,gt", [
    ("Qual o prazo da denuncia no CPP?", "art. 46 CPP - 5 dias preso / 15 solto"),
])
def test_pipeline_rag(question, gt):
    ans, ctxs = executar_pipeline(question)
    case = LLMTestCase(
        input=question, actual_output=ans,
        retrieval_context=ctxs, expected_output=gt,
    )
    assert_test(case, [
        FaithfulnessMetric(threshold=0.80, model=judge),
        AnswerRelevancyMetric(threshold=0.75, model=judge),
    ])
''')
with open(_arquivo, 'w', encoding='utf-8') as f: f.write(_codigo)
print(f'{_arquivo} gerado - execute com:  pytest {_arquivo} -v')


## Referências

CONFIDENT AI. **DeepEval — LLM evaluation framework**. <https://docs.confident-ai.com>.  
GROQ INC. **Groq API**. <https://console.groq.com/docs>.  
OLLAMA. **bge-m3 model**. <https://ollama.com/library/bge-m3>.
